# Dose response of the dependence labels (γ)

**Question.** Every exact label (S, N_r, g_w, Δm_k, σ) is measured at ONE dose: district k's probe transition at the
study's operating point. A foreign response at that dose can mean two different things: the gauge's signal moves *in
proportion* to k's demand change (dependence that scales with the drift), or k's drift tips the hub (pump switching,
tank cycling) and the gauge moves by a similar amount for *any* perturbation of k (a generic sensitivity of the hub, not
a dose-dependent dependence). On D-Town a large part of the off-diagonal shape effect looked like the second kind
(monthly S flat at 0.08–0.17 sd from 5 % to 100 % of the drift, 6 Oct hand-off); on KY7 it grew with the drift. This
notebook measures it on the probe stacks the placement uses, link by link.

**How.** For every drifting district k the drift is re-run at a fraction λ of its strength, with nothing else changed:

$$D_\lambda = D^{(0)} + \lambda\,\big(D^{(k)} - D^{(0)}\big)$$

($D^{(0)}$ the spliced no-drift city, $D^{(k)}$ probe world k's demand; only k's columns differ). λ = 1 is the probe
world itself and is re-run too, so every dose goes through the same EPANET loop (gate G-D2 checks it reproduces the
stored probe world). Each run is paired against the no-drift city on k's settled window — the exact layer's window.

The exponent is named γ because β is already the land-use level dial. Cost: K × |Λ| full-horizon runs per partition
(D-Town: 5 × 3, ≈ 85 s each on the study grid); the per-link results are cached, not the series.

### Definitions

For link s, drifting district k, dose λ, on k's settled window $W_k$ (hours), $\Delta Q_\lambda = Q^{(k,\lambda)}_s - Q^{(0)}_s$:

| quantity | definition | reading |
|---|---|---|
| $A_{s,k}(\lambda)$ | $\sqrt{\operatorname{mean}_{t\in W_k} \Delta Q_\lambda(t)^2}$ (L/s) | native response amplitude |
| $S_{s,k}(\lambda)$ | rms over the week of $z(PQ^{(k,\lambda)}) - z(PQ^{(0)})$ (as the exact layer) | what a client sees after per-client scaling |
| $\gamma_A$, $\gamma_S$ | least-squares slope of $\log A$ (resp. $\log S$) on $\log\lambda$ over $\lambda\in\Lambda$ | 1: the response grows in proportion to the drift (linear regime); 0: the same response at any dose (switch-like, hub timing); between: saturating |
| $\rho_S$ | $S(\lambda_{\min}) / (\lambda_{\min}\,S(1))$ | 1 proportional; $1/\lambda_{\min}$ dose-independent |
| resolved | $A(\lambda_{\min}) \ge$ `RES_REL` × mean$\lvert q_s\rvert$, `RES_REL` = 10 × the solver accuracy | below it the smallest dose's response is within the solver's tolerance and γ is not defined |
| responds | $S(1)\ge$ `S_floor` or $N_r(1)\ge$ `Nr_floor` (the taxonomy floors) | the cells that carry a dependence label |
| channel | E2 storage share σ at full dose: demand (σ < `ch_lo`), both, storage (σ > `ch_hi`), undefined (σ masked below `exact.sigma_floor`) | which mechanism carries the response |

Why it matters for the labels: a foreign label with $\gamma_S \approx 0$ says "k's drift switches the hub", and the same
shift appears for any perturbation of k, including ordinary variability; a label with $\gamma_S \approx 1$ scales with
how much k drifts. Un-biasing a client's data for its neighbours' drift (the FL use of these labels) treats the two
differently: the first is a step, the second a proportion. `GAMMA_MIN` (0.5) is the line the 6 Oct design used for a
dose-filtered shape label; it is a reading aid here, not a gate.

In [ ]:
%matplotlib inline
import warnings, logging, sys, pathlib, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from fedwater.config import load_base_params
from fedwater.experiments.spec import expand_study, load_studies
from fedwater.placement import epanet_loop as el
from fedwater.placement import exact as ex
from fedwater.placement import select as sel

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)
pd.set_option("display.width", 230, "display.max_columns", 60, "display.max_rows", 300)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})

STUDIES = ["dtown_districting_placement"]   # one built world per partition is enough: its stacks.yml names the stack
WORLDS = None                   # or {(network, partition): [sim_hash, ...]} to bypass the study expansion
STACKS = None                   # or [probe stack dir, ...]: skip the studies (no placed sensors then)
LAMBDAS = (0.03, 0.3, 1.0)      # dose: fraction of the probe drift; 1.0 is re-run and gated against the probe world
GAMMA_MIN = 0.5                 # a response "grows with the drift" when its dose exponent is >= this
FORCE = False                   # re-run dose worlds even when cached

BASE = load_base_params(REPO)
EXP_ROOT = REPO / load_studies(REPO).get("root", "data/09_experiments")
OUT = REPO / "notebooks" / "outputs" / "dose_response"
OUT.mkdir(parents=True, exist_ok=True)
EXACT_CFG = BASE["sensor_placement"].get("exact") or {}
TAX = {**sel.TAXONOMY_DEFAULTS, **(BASE["sensor_placement"].get("taxonomy") or {})}
FLOOR = {"S": float(TAX["S_floor"]), "Nr": float(TAX["Nr_floor"])}
ACC = float((BASE.get("hydraulics") or {}).get("accuracy") or 1e-5)
RES_REL = 10 * ACC              # a dosed response below RES_REL x the link's mean |q| is not resolved
GATE = {"pre_drift": 1e-6, "full": 1e-6, "branch_gain": 1e-3}
assert 1.0 in LAMBDAS and min(LAMBDAS) > 0
LAMS = sorted(LAMBDAS)
print(f"experiments {EXP_ROOT}  ->  {OUT} | lambdas {LAMS} | resolution {RES_REL:.0e} x mean|q|")

In [ ]:
# the probe stacks to dose: the LABEL stack of one built world per (network, partition) -- the stack the placed
# sensors' labels come from -- or the STACKS given
TARGETS = []
if STACKS:
    for p in map(Path, STACKS):
        mf = json.loads((p / "manifest.json").read_text())
        TARGETS.append({"key": (mf["network"], mf["method"]), "stack": p, "placed": None, "world": None})
else:
    if WORLDS is None:
        WORLDS = {}
        for st in STUDIES:
            for w in expand_study(st, REPO)["worlds"]:
                WORLDS.setdefault((w["network"], w["flat"]["districting"]), []).append(w["sim_hash"])
    for key, hashes in WORLDS.items():
        for h in sorted(set(hashes)):
            mf = json.loads((EXP_ROOT / "worlds" / h / "manifest.json").read_text())
            if mf["status"] != "ok":
                continue
            clone = EXP_ROOT / "worlds" / h / "clone"
            ptr = yaml.safe_load((clone / "data/08_reporting/sensor_placement/stacks.yml").read_text())
            pl = pd.read_csv(clone / "data/03_primary/sensor_placement.csv", dtype={"element": str})
            TARGETS.append({"key": key, "stack": Path(ptr["label"]["path"]), "placed": pl, "world": h})
            break
for t in TARGETS:
    t["X"] = ex.ensure_exact(t["stack"], EXACT_CFG)
    if not t["X"]["gates"]["passed"].all():
        raise ValueError(f"{t['key']}: the exact layer's gates fail -- dose results would not be trustworthy")
display(pd.DataFrame([{"network": t["key"][0], "partition": t["key"][1], "stack": t["stack"].name,
                       "exact": t["X"]["path"].name, "world (placement)": t["world"],
                       "drifting districts": len(t["X"]["manifest"]["districts"]),
                       "dose runs": len(t["X"]["manifest"]["districts"]) * len(LAMS)} for t in TARGETS]))

## Dose runs and their gates

| gate | check |
|---|---|
| G-D0 | every run reached the horizon and converged (no unbalanced step) |
| G-D1 | before k's demand departs from the no-drift city, the dosed flows are the no-drift city's (max relative deviation per link) |
| G-D2 | λ = 1, re-run through the same loop, reproduces the stored probe world over the whole horizon |
| G-D3 | on branches the response is exactly the downstream demand change, so the gain per unit of the DOSED demand change equals the full-dose gain at every λ (gain units) |

In [ ]:
# the dose worlds: D_lambda = D0 + lambda (D_k - D0), simulated through the exact layer's EPANET loop; per (k, lambda)
# the per-link response on k's settled window is cached (no series kept)
def stack_inputs(t):
    """No-drift city (spliced, and its flows), every probe world's own-district demand, the model."""
    src = ex._physics_owner(t["stack"])
    dy = yaml.safe_load((t["stack"] / "districts.yml").read_text())
    districts = {d: [str(n) for n in ns] for d, ns in dy["districts"].items()}
    doc = {"districts": districts, "assets": dy.get("assets") or {}}
    names = list(districts)
    own, keep = {}, {}

    def demands():
        for d in names:
            P = ex._world(src, d, doc, with_wn=(d == names[0]), flows=True)
            if d == names[0]:
                keep["wn"], keep["sd"] = P.wn, P.steps_day
            own[d] = (P.demand[[c for c in P.demand.columns if c in set(districts[d])]].copy(),
                      P.flows.astype(np.float32))           # the stored probe world, for G-D2
            yield d, P.demand
    D0, _ = ex.splice_nodrift(demands(), districts)
    jcols = [c for c in D0.columns if c != "month"]
    f = t["X"]["path"] / "nodrift_flows.parquet"
    if f.exists():
        R0 = pd.read_parquet(f)
    else:                                       # exact layer cached without save_series: simulate it here
        r0 = el.simulate(keep["wn"], D0, len(D0), pmin_nodes=jcols, dtype=np.float32)
        R0 = pd.DataFrame(r0["snap"], columns=r0["links"])
    return {"names": names, "D0": D0, "jcols": jcols, "own": own, "wn": keep["wn"], "sd": keep["sd"], "R0": R0}


def dose_run(I, k, lam, win):
    D0 = I["D0"]
    Dk_own, Qk_probe = I["own"][k]
    kc = list(Dk_own.columns)
    Dl = D0.copy()
    Dl[kc] = D0[kc].to_numpy(float) + lam * (Dk_own.to_numpy(float) - D0[kc].to_numpy(float))
    H = len(D0)
    r = el.simulate(I["wn"], Dl, H, pmin_nodes=I["jcols"], dtype=np.float32)
    links = list(I["R0"].columns)
    ix = [r["links"].index(l) for l in links]
    Q = r["snap"][:, ix].astype(float)
    R0 = I["R0"].to_numpy(float)
    dif = np.abs(Dk_own.to_numpy(float) - D0[kc].to_numpy(float)).max(axis=1)
    t_div = int(np.argmax(dif > 0)) if (dif > 0).any() else H
    f0, f1 = win
    dD = lam * (Dk_own.to_numpy(float).sum(axis=1) - D0[kc].to_numpy(float).sum(axis=1))[f0:f1]
    Q0w = R0[f0:f1]
    ef = ex.paired_effect(Q[f0:f1], Q0w, dD, I["sd"], f0, ex._orient(Q0w))
    dQ = Q[f0:f1] - Q0w
    out = pd.DataFrame({"element": links, "k": k, "lam": lam, "A": np.sqrt((dQ ** 2).mean(axis=0)),
                        "S": ef["S"], "g_w": ef["g_w"], "mean_abs_q0": np.abs(Q0w).mean(axis=0)})
    gate = {"k": k, "lam": lam, "completed": bool(r["completed"]),
            "unbalanced_steps": int(r["warnings"].get(1, 0)), "seconds": round(r["seconds"], 1), "t_div": t_div,
            "pre_drift_max_rel": float(np.nanmax(ex._rel_dev(Q[:t_div], R0[:t_div]))) if t_div else 0.0}
    if lam == 1.0:                              # the full dose must reproduce the stored probe world
        Qp = Qk_probe[links].to_numpy(float)[:H]
        gate["full_vs_probe_max_rel"] = float(np.nanmax(ex._rel_dev(Q, Qp)))
    return out, gate


DOSE, DGATE = [], []
for t in TARGETS:
    X = t["X"]
    cache = OUT / "cache" / t["key"][0] / t["key"][1] / X["path"].name
    cache.mkdir(parents=True, exist_ok=True)
    I = None
    for k in X["manifest"]["districts"]:
        win = tuple(X["manifest"]["windows"][k]["settled"])
        for lam in LAMS:
            fc, fg = cache / f"{k}__{lam:g}.parquet", cache / f"{k}__{lam:g}.json"
            if fc.exists() and fg.exists() and not FORCE:
                o, g = pd.read_parquet(fc), json.loads(fg.read_text())
            else:
                if I is None:
                    t0 = time.time()
                    I = stack_inputs(t)
                    print(f"{t['key']}: inputs ({time.time() - t0:.0f} s)", flush=True)
                o, g = dose_run(I, k, lam, win)
                o.to_parquet(fc, index=False)
                fg.write_text(json.dumps(g))
                print(f"  {k} lambda {lam:g}: {g['seconds']:.0f} s", flush=True)
            DOSE.append(o.assign(network=t["key"][0], partition=t["key"][1]))
            DGATE.append({**g, "network": t["key"][0], "partition": t["key"][1]})
    del I
DOSE = pd.concat(DOSE, ignore_index=True)
DOSE["element"] = DOSE["element"].astype(str)
DGATE = pd.DataFrame(DGATE)

# branch physics on every dose: a branch's gain per unit of the dosed demand change is the full-dose gain
br = []
for t in TARGETS:
    L = t["X"]["links"].assign(element=lambda d: d["element"].astype(str))
    B = set(L.loc[(L["topo"] == "branch") & ~L["dead"].astype(bool), "element"])
    d = DOSE[(DOSE["network"] == t["key"][0]) & (DOSE["partition"] == t["key"][1]) & DOSE["element"].isin(B)]
    g1 = d[d["lam"] == 1.0].set_index(["element", "k"])["g_w"]
    for lam, g in d.groupby("lam"):
        dev = (g.set_index(["element", "k"])["g_w"] - g1.reindex(g.set_index(["element", "k"]).index)).abs()
        br.append({"network": t["key"][0], "partition": t["key"][1], "lam": lam, "branch_max_abs_dg": float(dev.max())})
BR = pd.DataFrame(br)
G = [{"gate": "G-D0", "check": "every dose run reached the horizon, converged",
      "value": f"{int((~DGATE['completed'] | (DGATE['unbalanced_steps'] > 0)).sum())} / {len(DGATE)} failing",
      "passed": bool((DGATE["completed"] & (DGATE["unbalanced_steps"] == 0)).all())},
     {"gate": "G-D1", "check": "dosed flows = no-drift city before the drift (max rel)",
      "value": f"{DGATE['pre_drift_max_rel'].max():.1e}", "passed": bool(DGATE["pre_drift_max_rel"].max() <= GATE["pre_drift"])},
     {"gate": "G-D2", "check": "full dose (lambda 1) re-run = the stored probe world (max rel, whole horizon)",
      "value": f"{DGATE['full_vs_probe_max_rel'].max():.1e}",
      "passed": bool(DGATE["full_vs_probe_max_rel"].max() <= GATE["full"])},
     {"gate": "G-D3", "check": "branches: gain per unit dosed demand = full-dose gain (gain units)",
      "value": f"{BR['branch_max_abs_dg'].max():.1e}" if len(BR) else "no branches",
      "passed": bool((not len(BR)) or BR["branch_max_abs_dg"].max() <= GATE["branch_gain"])}]
DG = pd.DataFrame(G)
display(DG)
display(Markdown("per run"))
display(DGATE.set_index(["network", "partition", "k", "lam"]).round(8))
DOSE_OK = bool(DG["passed"].all())
display(Markdown(f"**Dose gates {'pass' if DOSE_OK else 'FAIL -- do not read the sections below'}**"))

## Dose exponents per cell

`dose_cells.parquet`: one row per (link, k) with A and S at every dose, γ_A, γ_S, ρ, and the exact labels at full dose
(S, N_r, g_w, Δm_k, σ, class, channel), topology, home, candidate / placed flags. `dose_runs.parquet`: every (link, k, λ).
The summary below counts responding candidate cells; rows per drifting district k sit beside the partition row.

In [ ]:
# per (link, drifting district k): the response at every dose, the dose exponents, and the exact labels at full dose
assert DOSE_OK, "a dose gate failed"
LMIN = LAMS[0]
CELLS = []
for t in TARGETS:
    X = t["X"]
    names = list(X["manifest"]["districts"])
    lab = X["labels"].assign(element=lambda d: d["element"].astype(str))
    lab["cls"] = sel.taxonomy(lab, TAX, "S")
    sig = lab["sigma"] if "sigma" in lab else pd.Series(np.nan, index=lab.index)
    lab["channel"] = np.select([sig < TAX["ch_lo"], sig > TAX["ch_hi"], sig.notna()],
                               ["demand", "storage", "both"], "undefined")     # E2 sigma at full dose
    L = X["links"].assign(element=lambda d: d["element"].astype(str)).set_index("element")
    d = DOSE[(DOSE["network"] == t["key"][0]) & (DOSE["partition"] == t["key"][1])]
    A = d.pivot_table(index=["element", "k"], columns="lam", values="A")
    Sx = d.pivot_table(index=["element", "k"], columns="lam", values="S")
    q0 = d[d["lam"] == 1.0].set_index(["element", "k"])["mean_abs_q0"]
    lx = np.log(np.array(LAMS))

    def slope(M):
        Y = np.log(M[LAMS].to_numpy(float))
        ok = np.isfinite(Y).all(axis=1)
        out = np.full(len(M), np.nan)
        xc = lx - lx.mean()
        out[ok] = (Y[ok] - Y[ok].mean(axis=1, keepdims=True)) @ xc / (xc @ xc)
        return out

    res = A[LMIN] >= RES_REL * q0.reindex(A.index)
    c = pd.DataFrame(index=A.index)
    c["gamma_A"] = np.where(res, slope(A.where(A > 0)), np.nan)
    c["gamma_S"] = np.where(res, slope(Sx.where(Sx > 0)), np.nan)
    c["rho_A"] = np.where(res, A[LMIN] / (LMIN * A[1.0]), np.nan)
    c["rho_S"] = np.where(res, Sx[LMIN] / (LMIN * Sx[1.0]), np.nan)
    c["resolved"] = res.to_numpy()
    for lam in LAMS:
        c[f"A_{lam:g}"], c[f"S_{lam:g}"] = A[lam], Sx[lam]
    c = c.reset_index().merge(lab[["element", "k", "own", "S", "Nr", "g_w", "dm_k", "sigma", "cls", "channel"]],
                              on=["element", "k"], how="left")
    c["topo"] = c["element"].map(L["topo"])
    c["home"] = c["element"].map(L["home"])
    c["candidate"] = c["element"].isin(sel.candidates(sel.prep(X, TAX)).index)
    c["responds"] = (c["S"] >= FLOOR["S"]) | (c["Nr"] >= FLOOR["Nr"])
    if t["placed"] is not None:
        pl = t["placed"].set_index("element")
        c["placed"] = c["element"].isin(pl.index)
        c["client"], c["slot"] = c["element"].map(pl["district"]), c["element"].map(pl["slot"])
        c["family"] = c["element"].map(pl["slot_class"])
    else:
        c["placed"], c["client"], c["slot"], c["family"] = False, None, None, None
    CELLS.append(c.assign(network=t["key"][0], partition=t["key"][1]))
CELLS = pd.concat(CELLS, ignore_index=True)
CELLS.to_parquet(OUT / "dose_cells.parquet", index=False)
DOSE.to_parquet(OUT / "dose_runs.parquet", index=False)

# the summary, split by drifting district (partition-level rows alongside, never instead)
LIVE = CELLS[CELLS["candidate"] & CELLS["responds"]]
rows = []
for (n, m, own), g in LIVE.groupby(["network", "partition", "own"]):
    for k, gk in [("all", g)] + list(g.groupby("k")):
        rows.append({"network": n, "partition": m, "cell": "own" if own else "foreign", "k": k,
                     "responding cells": len(gk), "resolved": int(gk["resolved"].sum()),
                     "median gamma_A": gk["gamma_A"].median(), "median gamma_S": gk["gamma_S"].median(),
                     f"share gamma_S < {GAMMA_MIN}": float((gk["gamma_S"] < GAMMA_MIN).sum() / max(gk["gamma_S"].notna().sum(), 1)),
                     "median rho_S": gk["rho_S"].median()})
SUM = pd.DataFrame(rows).set_index(["network", "partition", "cell", "k"])
display(Markdown(f"**Dose exponents of the responding candidate cells** (S >= {FLOOR['S']} or Nr >= {FLOOR['Nr']} "
                 f"at full dose; resolved: A({LMIN:g}) >= {RES_REL:.0e} x mean|q|). gamma 1: grows in proportion to "
                 f"the drift; 0: the same response at any dose. rho_S = S({LMIN:g}) / ({LMIN:g} S(1)): 1 proportional, "
                 f"{1 / LMIN:.0f} dose-independent."))
display(SUM.round(3))

## How the responses scale

Top: median and q10–q90 of A(λ)/A(1) and S(λ)/S(1) for responding, resolved cells. On the green dashed line the
response is proportional to the drift; on the dotted line it does not depend on the dose. Below: γ_S of the foreign
responding cells by channel (σ) and by topology, then per drifting district with the placed sensors in red.

In [ ]:
# how the response scales with the dose: A(lambda)/A(1) and S(lambda)/S(1), own vs foreign, per partition
COL = {True: "#1f77b4", False: "#d62728"}
parts = list(dict.fromkeys(zip(LIVE["network"], LIVE["partition"])))
for qty, lab_ in (("A", "native amplitude A (rms dQ)"), ("S", "shape effect S")):
    fig, axes = plt.subplots(1, len(parts), figsize=(8 * len(parts), 6.5), squeeze=False)
    for j, (n, m) in enumerate(parts):
        ax = axes[0, j]
        g = LIVE[(LIVE["network"] == n) & (LIVE["partition"] == m) & LIVE["resolved"]]
        for own in (True, False):
            q = g[g["own"] == own]
            if not len(q):
                continue
            R = q[[f"{qty}_{l:g}" for l in LAMS]].to_numpy(float) / q[[f"{qty}_1"]].to_numpy(float)
            ax.plot(LAMS, np.nanmedian(R, 0), marker="o", color=COL[own],
                    label=f"{'own district' if own else 'other districts'} (n = {len(q)})")
            ax.fill_between(LAMS, np.nanquantile(R, 0.1, 0), np.nanquantile(R, 0.9, 0), color=COL[own], alpha=0.15)
        ax.plot(LAMS, LAMS, color="#2ca02c", ls="--", lw=1, label="proportional (gamma = 1)")
        ax.axhline(1, color="0.4", ls=":", lw=1, label="dose-independent (gamma = 0)")
        ax.set(xscale="log", yscale="log", xlabel="dose lambda (fraction of the probe drift)",
               ylabel=f"{qty}(lambda) / {qty}(1)", title=f"{n} / {m} -- {lab_} (median, q10-q90; responding cells)")
        ax.legend(fontsize=8)
    fig.tight_layout(); plt.show()

# the dose exponent of the shape label, by channel and by topology (foreign responding cells)
order_ch = ["demand", "both", "storage", "undefined"]
fig, axes = plt.subplots(2, len(parts), figsize=(8 * len(parts), 11), squeeze=False)
rng = np.random.default_rng(0)
for j, (n, m) in enumerate(parts):
    g = LIVE[(LIVE["network"] == n) & (LIVE["partition"] == m) & ~LIVE["own"]]
    for i, (col, order) in enumerate((("channel", order_ch), ("topo", ["loop", "source_bridge", "branch"]))):
        ax = axes[i, j]
        data = [g.loc[g[col] == o, "gamma_S"].dropna().clip(-0.5, 1.5) for o in order]
        ax.boxplot(data, widths=0.6, showfliers=False)
        for x_, d_ in enumerate(data, start=1):
            ax.scatter(rng.normal(x_, 0.06, len(d_)), d_, s=6, alpha=0.35, color="#555")
        ax.axhline(1, color="#2ca02c", lw=0.8, ls="--"); ax.axhline(0, color="#d62728", lw=0.8, ls="--")
        ax.axhline(GAMMA_MIN, color="k", lw=0.6, ls=":")
        ax.set_xticks(range(1, len(order) + 1), [f"{o}\n(n = {len(d)})" for o, d in zip(order, data)])
        ax.set(ylim=(-0.55, 1.55), ylabel="gamma_S", title=f"{n} / {m} -- foreign responding cells by {col}")
fig.tight_layout(); plt.show()

# per drifting district: the foreign responding cells' gamma_S, as a link x k picture (placed sensors framed)
for (n, m) in parts:
    g = LIVE[(LIVE["network"] == n) & (LIVE["partition"] == m) & ~LIVE["own"]]
    names = sorted(g["k"].unique())
    fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4.6), squeeze=False, sharey=True)
    for j, k in enumerate(names):
        ax = axes[0, j]
        q = g[g["k"] == k]
        ax.hist(q["gamma_S"].dropna().clip(-0.5, 1.5), bins=np.linspace(-0.5, 1.5, 41), color="0.6",
                label=f"candidates (n = {q['gamma_S'].notna().sum()})")
        qp = q[q["placed"]]
        if len(qp):
            ax.hist(qp["gamma_S"].dropna().clip(-0.5, 1.5), bins=np.linspace(-0.5, 1.5, 41), color="#a50f15",
                    label=f"placed (n = {qp['gamma_S'].notna().sum()})")
        ax.axvline(GAMMA_MIN, color="k", lw=0.8, ls="--")
        ax.set(xlabel="gamma_S", title=f"{k.replace('District_', '')} drifts")
        ax.legend(fontsize=7)
    fig.suptitle(f"{n} / {m} -- dose exponent of the shape label, foreign responding cells, per drifting district")
    fig.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

## The placed sensors

Every foreign cell of a placed sensor that responds at full dose, with its dose exponents; then, per client, how many of
those labels grow with the drift (γ_S ≥ `GAMMA_MIN`) and how many are undefined (unresolved at the smallest dose).

In [ ]:
# the placed sensors: every foreign cell that responds at full dose, with its dose exponents
P = CELLS[CELLS["placed"] & ~CELLS["own"] & CELLS["responds"]]
if not len(P):
    display(Markdown("No placed sensors (STACKS mode) or no responding foreign cell on them."))
else:
    cols = ["client", "slot", "element", "k", "cls", "channel", "S", "gamma_S", "rho_S", "A_1", "gamma_A", "dm_k"]
    for (n, m), g in P.groupby(["network", "partition"]):
        display(Markdown(f"**{n} / {m} -- placed sensors, foreign cells that respond at full dose**"))
        display(g[cols].sort_values(["client", "slot", "k"]).round(3).set_index(["client", "slot"]))
        keep = g["gamma_S"] >= GAMMA_MIN
        t = g.assign(kept=keep).groupby("client").agg(
            foreign_responding=("k", "size"), grows_with_dose=("kept", "sum"),
            undefined=("gamma_S", lambda s: int(s.isna().sum())))
        display(Markdown(f"Per client: foreign responding cells, and how many keep a shape label whose dose exponent "
                         f"is >= {GAMMA_MIN} (the label grows with the drift)"))
        display(t)